In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Tutorial 12 — Settling flexible fiber (Delmotte 2015, Fig. 10)

A planar flexible fiber settles under transverse gravity. After a
transient "W" shape it converges toward a horseshoe-like steady
configuration whose curvature is set by the **elasto-gravitational
number** $B = F_\perp L / K_b$ (Eq. 55 of Delmotte, Climent &
Plouraboué, *J. Comput. Phys.* **286** (2015) 14–37).

We produce two figures:

1. **Settling transient** at fixed $B = 1000$ — five snapshots of an
   initially straight fiber, stacked vertically as time advances.
2. **Final shape vs rigidity** — the late-time configuration for
   four bending rigidities, $B \in \{6, 50, 200, 1000\}$, also
   stacked vertically (stiff → floppy).

The motion is planar so we use `FlexibleFiber(planar=True)`; the
fiber bends in the xz-plane and gravity acts along $-\boldsymbol{e}_3$.


In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import softmobility as sm
from softmobility.classes import figstyle

np.set_printoptions(precision=3, suppress=True)

# Apply the paper-figure matplotlib aesthetic (white background, black
# box, Helvetica labels at 11 pt, fixed colour palette). Every figure
# inherits it.
figstyle.apply()
FIGDIR = "figures"


## Helper — lab-frame bead positions

`FlowBodyRollout.rollout` returns the **body-reference** position and
orientation plus the per-bead DOFs. To plot the fiber shape we convert each
bead's body-frame position into the lab frame using the body Rodrigues vector.


In [2]:
def lab_bead_positions(fiber, positions, orientations, dofs, design=None):
    """Return an array of shape (n_steps, n_beads, 3) with lab-frame bead positions."""
    if design is None:
        design = np.asarray(fiber.design_defaults)
    n_steps = int(positions.shape[0])
    n_beads = fiber.Nspheres
    t_dummy = np.array([0.0])
    out = np.zeros((n_steps, n_beads, 3))
    body_frame_positions = np.zeros((n_beads, 3))
    for step in range(n_steps):
        dof_step = np.asarray(dofs[step])
        for i in range(n_beads):
            body_frame_positions[i] = np.asarray(fiber.spheres[i].position(dof_step, design, t_dummy))
        R = np.asarray(sm.rotation_matrix(orientations[step]))
        out[step] = np.asarray(positions[step]) + body_frame_positions @ R.T
    return out


## Note on the bending model

`FlexibleFiber` implements the Gears Model of Delmotte et al. 2015 (§2.3)
with a **linearized** version of the paper's bending torque (eq. 32 + 34
expanded to first order in joint angles, §2.4). Bond length is exactly
``2a`` (sphere surfaces touching). The bending torque on bead $i$ is

$$ \boldsymbol\gamma^b_i \;=\; \frac{K_b}{2 a}\,\bigl(\mathrm{rod}_{i-1} - 2\,\mathrm{rod}_i + \mathrm{rod}_{i+1}\bigr) $$

(adjacent-bead Laplacian on the orientation DOFs; first-difference at
the chain ends). This linearization keeps the bending operator linear in
the DOFs, which preserves the framework's exact `M_K @ dofs` interface
and its JIT-friendly tensor extraction.

**Approximation regime and known limitation.** This linearized model is
quantitatively accurate for joint angles ≲ π/4. It does *not* fully
capture the full $B = 10\,000$ horseshoe of the paper's Fig 10b, which
requires the geometric (non-linear) curvature formula (eq. 33 / 34) and
explicit no-slip Lagrange-multiplier constraints. We therefore stay
within $B \le 2000$ where the linear model remains stable.


## Settling fiber

### Elasto-gravitational number

$$B = \frac{F_\perp L}{K_b}, \qquad F_\perp = N\,m\,g.$$

Large $B$ means a floppy fiber. Set $m = 1$, $g = 1$, $N = 10$, and
solve for $K_b$ given a target $B$.


In [3]:
N_set = 10
a = 1.0
m = 1.0
g = 1.0

L_set = (N_set - 1) * 2 * a   # bond length is exactly 2a (Gears Model)
F_perp = N_set * m * g

def K_b_for_B(B: float) -> float:
    """Bending modulus that yields elasto-gravitational number ``B``."""
    return F_perp * L_set / B

# Build the fiber once at the B = 1000 reference. The bending modulus is a
# design parameter, so we can override it per-rollout via ``design=`` to
# sweep B without rebuilding (and re-JIT-compiling) the rollout.
B_ref = 1000.0
fiber_set = sm.FlexibleFiber(
    n_beads=N_set,
    radius=a,
    bending_rigidity=K_b_for_B(B_ref),
    mass=m,
    planar=True,
)
rollout_set = sm.FlowBodyRollout(
    soft_body=fiber_set,
    flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=g)},
)
print(f"L = {L_set:.1f},  F_perp = {F_perp:.1f},  K_b(B={B_ref:g}) = {K_b_for_B(B_ref):.4f}")


L = 18.0,  F_perp = 10.0,  K_b(B=1000) = 0.1800


### Run the rollout

The fiber is initially horizontal (along $\boldsymbol{e}_1$) and gravity pulls it
in $-\boldsymbol{e}_3$. The terminal settling speed is empirically $V_s\!\sim\!0.13$
in our units, so $\tau = L/V_s \approx 150$. We integrate over a few $\tau$
to see the W shape develop and the U-curl progress.

The first call triggers JAX/XLA compilation (~10–15 s); subsequent calls
(used in the rigidity sweep below) are sub-second because they reuse the
same JIT graph and only swap the design vector.


In [4]:
dt_set = 0.5
n_steps_set = 1500   # final time = 750 ≈ 5 τ

@jax.jit
def run_settling(design):
    """JIT-compiled rollout. Compiled once; reused for every B in the sweep."""
    return rollout_set.rollout(dt=dt_set, n_steps=n_steps_set, design=design)

design_ref = jnp.array([a, K_b_for_B(B_ref), m])
positions_g, orientations_g, dofs_g = run_settling(design_ref)
lab_pos_g = lab_bead_positions(
    fiber_set, positions_g, orientations_g, dofs_g, design=np.asarray(design_ref)
)
print("max |dof| over trajectory:", float(np.max(np.abs(dofs_g))))


max |dof| over trajectory: 2.000236988067627


### Figure 1 — Settling transient

Five snapshots of the $B = 1000$ fiber, COM-centred and stacked
vertically (top = early, bottom = late). Beads are drawn as
filled circles at the true bead radius $a$ — the visible bead
size relative to the fiber length is $a / L = 1/18$.


In [ ]:
def draw_shape(ax, beads_xz, *, color, fill, line_width=1.5, bead_radius=a, opacity=1.0):
    """Draw a fiber as a connecting line plus one filled circle per bead."""
    xs, zs = beads_xz[:, 0], beads_xz[:, 1]
    ax.plot(xs, zs, color=color, linewidth=line_width, alpha=opacity)
    for cx, cz in zip(xs, zs, strict=True):
        ax.add_patch(plt.Circle(
            (cx, cz), bead_radius,
            edgecolor=color, facecolor=fill, linewidth=1.0, alpha=opacity,
        ))


def stack_shape(snapshot_xz, row_index, z_step):
    """Centre an (n_beads, 2) snapshot at the origin, then shift down by row_index*z_step."""
    centred = snapshot_xz - snapshot_xz.mean(axis=0)
    centred[:, 1] -= row_index * z_step
    return centred


step_indices = np.array([0, 100, 300, 700, 1499])
times = step_indices * dt_set
z_step = 0.55 * L_set

# Layout. Bead column extends to roughly ±0.5 L; reserve the leftmost
# 0.6 L of the canvas for labels.
x_label = -0.8 * L_set
x_left, x_right = -.85 * L_set, 0.55 * L_set

fig1, ax1 = figstyle.figure(size="half", aspect=0.6)

for i, (step, t) in enumerate(zip(step_indices, times, strict=True)):
    snap_xz = lab_pos_g[step][:, [0, 2]]      # take (x, z)
    placed = stack_shape(snap_xz, i, z_step)
    draw_shape(
        ax1, placed,
        color=figstyle.COLORS["red"],
        fill=figstyle.COLORS["red_25"],
        opacity=0.6,
    )
    figstyle.add_label(
        ax1, (x_label, -i * z_step), f"t = {t:.0f}", anchor="middle right",
    )

y_top = 0.5 * z_step
y_bot = -(len(step_indices) - 1) * z_step - 0.5 * z_step
ax1.set_xlim(x_left, x_right)
ax1.set_ylim(y_bot, y_top)
ax1.set_aspect("equal")
ax1.set_axis_off()

figstyle.save(fig1, "fig_settling_transient", figdir=FIGDIR)
fig1

### Figure 2 — Final shape vs bending rigidity

Late-time configurations for four values of $B$, stacked vertically
(top = stiff, bottom = floppy). At $B = 6$ the fiber barely deforms;
by $B = 1000$ it has folded into a tight U-curl. The most flexible
cases relax slowly from a straight fiber, so we **warm-start** each
rollout from the converged shape of the previous (less-flexible) $B$
— this gives a converged equilibrium for every $B$ at a fraction of
the cost of running each one cold from the straight chain.


In [ ]:
# Warm-start chain: run B values in order of increasing flexibility,
# passing each run's final state as the IC for the next. The most
# flexible cases (large B) take the longest to relax from a straight
# fiber; warm-starting them from the previous (less-flexible) shape
# cuts the transient. n_steps_chain is set so each leg converges to
# Δ ~ 1e-3 a (see drafts/test_rigidity_sweep_convergence.py).
n_steps_chain = 3000

@jax.jit
def run_settling_chain(design, init_position, init_orientation, init_dofs):
    """JIT rollout that accepts a starting state for warm-start chaining."""
    return rollout_set.rollout(
        dt=dt_set, n_steps=n_steps_chain, design=design,
        init_position=init_position,
        init_orientation=init_orientation,
        init_dofs=init_dofs,
    )

B_values = [6.0, 50.0, 200.0, 1000.0]
final_shapes = []
max_dof_log = []

# Cold start IC: straight fiber at origin (the framework defaults).
init_position = jnp.zeros(3)
init_orientation = jnp.zeros(3)
init_dofs = jnp.zeros(fiber_set.Ndof)

for B in B_values:
    design_B = jnp.array([a, K_b_for_B(B), m])
    pos, ori, dof = run_settling_chain(
        design_B, init_position, init_orientation, init_dofs,
    )
    final_shapes.append(
        lab_bead_positions(fiber_set, pos, ori, dof, design=np.asarray(design_B))[-1]
    )
    max_dof_log.append((B, float(np.max(np.abs(dof)))))
    # Carry the converged state forward as the IC for the next (more flexible) B.
    init_position = pos[-1]
    init_orientation = ori[-1]
    init_dofs = dof[-1]

print("max |dof| per B:")
for B, m_dof in max_dof_log:
    print(f"  B = {B:>6g}    max|dof| = {m_dof:.3f}")

fig2, ax2 = figstyle.figure(size="half", aspect=0.6)

for i, (B, snap) in enumerate(zip(B_values, final_shapes, strict=True)):
    snap_xz = snap[:, [0, 2]]
    placed = stack_shape(snap_xz, i, z_step)
    draw_shape(
        ax2, placed,
        color=figstyle.COLORS["red"],
        fill=figstyle.COLORS["red_25"],
        opacity=0.6,
    )
    figstyle.add_label(
        ax2, (x_label, -i * z_step), f"B = {B:g}", anchor="middle right",
    )

y_bot2 = -(len(B_values) - 1) * z_step - 0.5 * z_step
ax2.set_xlim(x_left, x_right)
ax2.set_ylim(y_bot2, y_top)
ax2.set_aspect("equal")
ax2.set_axis_off()

figstyle.save(fig2, "fig_settling_rigidity_sweep", figdir=FIGDIR)
fig2

## Notes

* `FlexibleFiber` uses **rigid bonds** (paper Joint Model, Eqs. 2–4)
  parameterised by per-bead orientation DOFs, plus a **linear discrete
  biharmonic** bending torque. This captures the qualitative shapes of
  Fig. 10 (W shape, U-curl) but does **not** bound the per-joint
  angle, so the full $B = 10\,000$ horseshoe of Fig. 10b would require
  the geometric curvature formula (paper Eq. 33) to stay stable —
  out of scope for this tutorial.
* The rigidity sweep uses a **warm-start chain**: each rollout starts
  from the converged final state of the previous $B$ via the
  ``init_position`` / ``init_orientation`` / ``init_dofs`` arguments
  of ``rollout``. The chain JIT is compiled once and reused for every
  $B$ in the sweep — only the design vector and the IC change between
  rollouts.
